In [2]:
import os

# Set working directory to notebooks/ folder every time
os.chdir(r'C:\Users\pc\OneDrive\Desktop\credit-risk-dashboard\notebooks')

print("Working directory:", os.getcwd())

Working directory: C:\Users\pc\OneDrive\Desktop\credit-risk-dashboard\notebooks


In [3]:
import duckdb
import pandas as pd

# Load CSV
df = pd.read_csv(r"C:\Users\pc\OneDrive\Desktop\credit-risk-dashboard\Data\Give me some credit.csv", index_col=0)

# Rename columns immediately
df.rename(columns={
    'SeriousDlqin2yrs'                             : 'is_default',
    'RevolvingUtilizationOfUnsecuredLines'          : 'revolving_util',
    'age'                                          : 'age',
    'NumberOfTime30-59DaysPastDueNotWorse'         : 'late_30_59',
    'DebtRatio'                                    : 'debt_ratio',
    'MonthlyIncome'                                : 'monthly_income',
    'NumberOfOpenCreditLinesAndLoans'              : 'open_credit_lines',
    'NumberOfTimes90DaysLate'                      : 'late_90_plus',
    'NumberRealEstateLoansOrLines'                 : 'real_estate_loans',
    'NumberOfTime60-89DaysPastDueNotWorse'         : 'late_60_89',
    'NumberOfDependents'                           : 'num_dependents'
}, inplace=True)

# Add a row ID
df.reset_index(inplace=True)
df.rename(columns={'index': 'customer_id'}, inplace=True)

print(f"Shape: {df.shape}")
print(df.head(3))

Shape: (150000, 12)
   ID  is_default  revolving_util  Age  late_30_59  debt_ratio  \
0   1           1        0.766127   45           2    0.802982   
1   2           0        0.957151   40           0    0.121876   
2   3           0        0.658180   38           1    0.085113   

   monthly_income  open_credit_lines  late_90_plus  real_estate_loans  \
0          9120.0                 13             0                  6   
1          2600.0                  4             0                  0   
2          3042.0                  2             1                  0   

   late_60_89  num_dependents  
0           0             2.0  
1           0             1.0  
2           0             0.0  


In [4]:
# Connect (creates file if it doesn't exist)
con = duckdb.connect('../data/credit_risk.db')

# Write raw table
con.execute("DROP TABLE IF EXISTS raw_loans")
con.execute("CREATE TABLE raw_loans AS SELECT * FROM df")

# Verify
count = con.execute("SELECT COUNT(*) FROM raw_loans").fetchone()[0]
print(f"✅ raw_loans created with {count:,} rows")

# Preview
print(con.execute("SELECT * FROM raw_loans LIMIT 3").df())

con.close()

✅ raw_loans created with 150,000 rows
   ID  is_default  revolving_util  Age  late_30_59  debt_ratio  \
0   1           1        0.766127   45           2    0.802982   
1   2           0        0.957151   40           0    0.121876   
2   3           0        0.658180   38           1    0.085113   

   monthly_income  open_credit_lines  late_90_plus  real_estate_loans  \
0          9120.0                 13             0                  6   
1          2600.0                  4             0                  0   
2          3042.0                  2             1                  0   

   late_60_89  num_dependents  
0           0             2.0  
1           0             1.0  
2           0             0.0  


In [5]:
con = duckdb.connect('../data/credit_risk.db')

print("=== TABLES ===")
print(con.execute("SHOW TABLES").df())

print("\n=== PREVIEW ===")
print(con.execute("SELECT * FROM raw_loans LIMIT 3").df())

print("\n=== DEFAULT RATE ===")
print(con.execute("""
    SELECT is_default, COUNT(*) as count
    FROM raw_loans 
    GROUP BY is_default
""").df())

con.close()


=== TABLES ===
        name
0  raw_loans

=== PREVIEW ===
   ID  is_default  revolving_util  Age  late_30_59  debt_ratio  \
0   1           1        0.766127   45           2    0.802982   
1   2           0        0.957151   40           0    0.121876   
2   3           0        0.658180   38           1    0.085113   

   monthly_income  open_credit_lines  late_90_plus  real_estate_loans  \
0          9120.0                 13             0                  6   
1          2600.0                  4             0                  0   
2          3042.0                  2             1                  0   

   late_60_89  num_dependents  
0           0             2.0  
1           0             1.0  
2           0             0.0  

=== DEFAULT RATE ===
   is_default   count
0           0  139974
1           1   10026
